#Main Preprocessing

##Fetching & Looking at Data

In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import euclidean_distances
import os

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

# column index
leg_length,torso_length,shoulder_width,hip_width = 0,1,2, 3

class CombinedAttributesAdder(BaseEstimator, TransformerMixin):
    def __init__(self, val=True): # no *args or **kargs
        self.val = val
    def fit(self, X, y=None):
        return self  # nothing else to do
    def transform(self, X):
        leg_torso= X[:,leg_length] / X[:,torso_length]
        leg_shoulder=X[:,leg_length] / X[:,shoulder_width]
        leg_hip=X[:,leg_length] / X[:,hip_width]
        torso_shoulder=X[:,torso_length] / X[:,shoulder_width]
        torso_hip=X[:,torso_length] / X[:,hip_width]
        shoulder_hip=X[:,shoulder_width] / X[:,hip_width]
        new_features = np.c_[
            leg_torso,
            leg_shoulder,
            leg_hip,
            torso_shoulder,
            torso_hip,
            shoulder_hip
        ]
        return np.c_[new_features] if self.val else X


In [ ]:
def final_func(path):
  #imports dataset
  clothing_num_data=pd.read_csv(path)
  metrics=clothing_num_data.drop(["image_name"], axis=1)
  clothes=clothing_num_data.drop(list(metrics.columns),axis=1)



  #Simple Imputer for median calculations
  imputer1 = SimpleImputer(strategy="median")
  imputer1.fit(metrics)
  X=imputer1.transform(metrics)
  metrics_f=pd.DataFrame(
      X,
      columns=metrics.columns,
      index=metrics.index
  )
  # print(metrics_f,'\n')



  # ratio calculated
  attr_adder = CombinedAttributesAdder()
  metrics_extra_attribs = attr_adder.transform(metrics_f.values)
  metrics_real_attribs = pd.DataFrame(
      metrics_extra_attribs,
      columns=["leg_torso","leg_shoulder","leg_hip","torso_shoulder","torso_hip","shoulder_hip"])
  # metrics_extra_attribs.head()
  # print(metrics_real_attribs,'\n')


  scaler = StandardScaler()
  final_metrics = scaler.fit_transform(metrics_real_attribs)
  # print("final_metrics",final_metrics)
  dataset_f = pd.DataFrame(
    final_metrics,
    columns=["leg_torso","leg_shoulder","leg_hip","torso_shoulder","torso_hip","shoulder_hip"])
  # print(dataset_f)
  return dataset_f,clothes,scaler

In [ ]:
dataset_f,clothes,scaler=final_func('Females_img_details.csv')
# print(dataset_f,clothes)
# print(type(clothes))

# xyz,abcd=final_func('img_details.csv')
# print(xyz,abcd)

In [ ]:
scaler

StandardScaler()

#Store The Preprocessed Data

In [ ]:
final_dataset=dataset_f

NameError: name 'dataset_f' is not defined

#KMeans Models

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

In [ ]:
kmeans = KMeans(n_clusters=4)
kmeans.fit(final_dataset)

KMeans(n_clusters=4)

In [ ]:
final_dataset['group_no.']=kmeans.labels_

#User Metrics Standardization

In [ ]:
def user_data_prep(path,scaler):
  #imports dataset
  clothing_num_data=pd.read_csv(path)

  metrics=clothing_num_data.drop(["image_name"], axis=1)
  clothes=clothing_num_data.drop(list(metrics.columns),axis=1)


  #Simple Imputer for median calculations
  imputer1 = SimpleImputer(strategy="median")
  imputer1.fit(metrics)
  X=imputer1.transform(metrics)
  metrics_f=pd.DataFrame(
      X,
      columns=metrics.columns,
      index=metrics.index
  )
  # print(metrics_f,'\n')



  # ratio calculated
  attr_adder = CombinedAttributesAdder()
  metrics_extra_attribs = attr_adder.transform(metrics_f.values)
  metrics_real_attribs = pd.DataFrame(
      metrics_extra_attribs,
      columns=["leg_torso","leg_shoulder","leg_hip","torso_shoulder","torso_hip","shoulder_hip"])
  # metrics_extra_attribs.head()
  # print(metrics_real_attribs,'\n')


  scaler = scaler
  final_metrics = scaler.transform(metrics_real_attribs)
  # print("final_metrics",final_metrics)
  dataset_f = pd.DataFrame(
    final_metrics,
    columns=["leg_torso","leg_shoulder","leg_hip","torso_shoulder","torso_hip","shoulder_hip"])
  # print(dataset_f)
  return dataset_f,clothes

In [ ]:
user_metrics,userimg = user_data_prep('user_data.csv',scaler)

FileNotFoundError: [Errno 2] No such file or directory: 'user_data.csv'

#User Dataset and Final Dataset Prepared and clustered till here

#From here creating the dataset for the similarity calculations

In [ ]:
final_dataset_copy = final_dataset

In [ ]:
centroids = kmeans.cluster_centers_

# New data point (same number of features as the original dataset)
new_data = user_metrics.values

# Calculate the Euclidean distance from the new data point to each centroid
distances = [np.linalg.norm(new_data - centroid) for centroid in centroids]
distances
# # Get the indices of the top 3 closest centroids based on the smallest distances
top_3_clusters = np.argsort(distances)[:3]
top_3_clusters
# # Print the top 3 closest clusters
# print("Top 3 closest clusters to the new data point:")
arr=[]
for cluster_idx in top_3_clusters:
    arr.append(cluster_idx)
    # print(f"Cluster {cluster_idx}: Distance = {distances[cluster_idx]:.2f}")


In [ ]:
def clustered_set(model_dataset,testing_set,num):
  temp_set=model_dataset[model_dataset['group_no.']==num]
  testing_set=pd.concat([testing_set,temp_set])
  model_dataset=model_dataset.drop(temp_set.index)
  print(testing_set,'\n')
  print(model_dataset,'\n')
  return model_dataset,testing_set


In [ ]:
testing_set=pd.DataFrame()
for num in arr:
  final_dataset_copy,testing_set=clustered_set(final_dataset_copy,testing_set,num)

#Till here the Testing_set for created and now similarity can be applied

##Euclid Recommendation

In [ ]:
def euclid_func(testing_set,user_metrics):
  euclid_dists = euclidean_distances(testing_set.drop('group_no.',axis=1), user_metrics).flatten()
  euclid_dists_points = np.argsort(euclid_dists)
  testing_set.iloc[euclid_dists_points[:5]].index
  euclidean_indices=np.array(testing_set.iloc[euclid_dists_points[:5]].index)
  if !os.path.isfile('euclid_recommends.csv'):
    clothes.iloc[euclidean_indices].to_csv('euclid_recommends.csv')
  else:
    clothes.iloc[euclidean_indices].to_csv('euclid_recommends.csv', mode='a', header=False)
  return None

In [ ]:
euclid_func(testing_set,user_metrics)

##Cosine Similarity (incase dataset is too large and the backend part takes too long to run)

In [ ]:
# testing_set, user_metrics

In [ ]:
# from sklearn.metrics.pairwise import cosine_similarity

# # Compute cosine similarity
# cos_similarities = cosine_similarity(testing_set.drop('group_no.',axis=1), user_metrics).flatten()

# # Sort in descending order (higher similarity is better)
# nearest_indices_cos = np.argsort(-cos_similarities)[:4]
# # closest_clothes_cos = labels[nearest_indices_cos]

# # Display results
# print("\nClosest matches using Cosine Similarity:")
# for idx in nearest_indices_cos:
#     print(f"Similarity: {cos_similarities[idx]:.4f}")

##Code After Applying PCA

In [ ]:
# copy_for_PCA,user_metrics_copy=testing_set.drop('group_no.',axis=1),user_metrics

In [ ]:
# copy_for_PCA,user_metrics_copy

In [ ]:
# from sklearn.decomposition import PCA

# # Reduce dimensions to 2 for faster calculations
# pca = PCA(n_components=3)

# features_reduced = pca.fit_transform(copy_for_PCA)
# user_ratios_reduced = pca.transform(user_metrics_copy)

# # Compute distances in reduced space
# distances_reduced = euclidean_distances(features_reduced, user_ratios_reduced).flatten()


In [ ]:
# distances_reduced

In [ ]:
# distances_reduced-euclid_dists

In [ ]:
# r=(distances_reduced-euclid_dists)/euclid_dists

#Experimentation

In [ ]:
final_dataset

,leg_torso,leg_shoulder,leg_hip,torso_shoulder,torso_hip,shoulder_hip,group_no.
0,-1.439944,0.995348,1.201599,2.477876,2.506892,0.421438,2
1,1.571095,3.761539,3.296917,2.414493,1.992229,-0.129653,2
2,-0.082723,1.111809,1.019512,1.225805,1.066430,-0.006345,2
3,-0.945015,1.545580,1.960623,2.579289,2.871267,0.742904,2
4,-0.421118,1.208602,1.180339,1.637377,1.501174,0.098992,2
...,...,...,...,...,...,...,...
459,0.053623,-0.648879,-0.975741,-0.752979,-1.051849,-0.823601,3
460,-0.663969,-0.581070,-0.893581,-0.205740,-0.595486,-0.769801,3
461,-0.489248,-0.872999,-1.014704,-0.661468,-0.824449,-0.516029,3
462,1.534503,0.822647,0.165456,-0.180827,-0.656041,-0.913436,0


In [ ]:
# clothes.to_csv('Females_clothes.csv')